In [ ]:
import math

In [ ]:
# NNs are large mathematical expressions.
# We create data structure that maintains this expression

class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0     # This is derivative of final output w.r.t self (this object)

        # This func is empty for those who does not formed by performing operation i.e leaf node
        self._backward = lambda: None

        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'Value(data={self.data})'
    
    # operation methods
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        # It will calculate grad for each child of current object using chain rule
        # Means grad of child1 w.r.t L & grad of child2 w.r.t L
        def _backward():
            # for f(x,y) = x + y , df/dx = 1 & df/dy = 1
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # for f(x,y) = x * y, df/dx = y & df/dy = x
            self.grad += other.data * out.grad
            other.grad += self.data  * out.grad
        
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only support int/float values as power"

        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * (self.data**(other - 1))) * out.grad
        out._backward = _backward
        return out
    
    # a/b = a * b**-1.
    def __truediv__(self, other):
        return self * other**-1
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    # fallback function for __add__() & __mul__()
    def __radd__(self, other):
        return self + other
    def __rmul__(self, other):
        return self * other
    
    # tanh operation
    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # derivative of tanh(x) is 1 - tanh(x)^2
            self.grad += (1 - (t**2)) * out.grad
        
        out._backward = _backward
        return out
    
    # exponent operation
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    
    
    def backward(self):

        # It sets all instances in sequential order which resulted in self. Meaning, it finds all its childs nodes and sets them in sequential manner i.e topological graph (graph that only flows from left  to right)
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        # build topological graph
        build_topo(self)

        # set grad of L w.r.t itself (l) to 1. because initially, we set to 0.0
        self.grad = 1
        
        # calculate grads
        for node in reversed(topo):
            node._backward()
        
